# 2. TF-IDF Representation & Baseline Evaluation
**Nova IMS — Text Mining 2025/2026**

This notebook implements the Bag-of-Words TF-IDF vectorization and compares the performance of:
1. **TF-IDF Unigrams** (`ngram_range=(1,1)`)
2. **TF-IDF Unigrams + Bigrams** (`ngram_range=(1,2)`)

Both representations are evaluated using a **Logistic Regression** baseline classifier with class weights balanced to handle target sentiment distribution.

In [ ]:
import os
import sys
# Ensure project src is in the system path
sys.path.append(os.path.abspath('..'))

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from src.train_val_split import stratified_split
from src.preprocessing import preprocess_tweet
from src.evaluate import evaluate_and_log

## 💾 1. Load Data & Perform Split

In [ ]:
train_df = pd.read_csv('../data/train.csv')
X_train, X_val, y_train, y_val = stratified_split(train_df)
print(f"Train set size: {len(X_train)} | Validation set size: {len(X_val)}")

## 🧹 2. Apply Custom Preprocessing
We preprocess the text using the custom pipeline from `src/preprocessing.py`, utilizing WordNet Lemmatization while preserving placeholders.

In [ ]:
print("Preprocessing training set...")
X_train_preprocessed = X_train.apply(lambda t: preprocess_tweet(t, return_str=True))
print("Preprocessing validation set...")
X_val_preprocessed = X_val.apply(lambda t: preprocess_tweet(t, return_str=True))
print("Preprocessing complete!")

## 🔬 3. Feature Extraction (TF-IDF)
We fit both vectorizers strictly on the **training set only** to prevent data leakage.

In [ ]:
# TF-IDF Unigrams
vec_uni = TfidfVectorizer(ngram_range=(1, 1))
X_train_uni = vec_uni.fit_transform(X_train_preprocessed)
X_val_uni = vec_uni.transform(X_val_preprocessed)
vocab_size_uni = len(vec_uni.vocabulary_)

# TF-IDF Unigrams + Bigrams
vec_bi = TfidfVectorizer(ngram_range=(1, 2))
X_train_bi = vec_bi.fit_transform(X_train_preprocessed)
X_val_bi = vec_bi.transform(X_val_preprocessed)
vocab_size_bi = len(vec_bi.vocabulary_)

print(f"Unigrams Vocab size: {vocab_size_uni}")
print(f"Unigrams + Bigrams Vocab size: {vocab_size_bi}")
print(f"Vocabulary size grew by {vocab_size_bi / vocab_size_uni:.2f}x when adding bigrams.")

## ⚖️ 4. Model Baseline Evaluation (Logistic Regression)

### Model A: Unigrams Baseline

In [ ]:
lr_uni = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_uni.fit(X_train_uni, y_train)
y_pred_uni = lr_uni.predict(X_val_uni)

# Evaluate and log metrics to outputs/results.csv
metrics_uni = evaluate_and_log(
    y_val, y_pred_uni,
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,1)",
    params="max_iter=1000, class_weight='balanced', random_state=42"
)

### Model B: Unigrams + Bigrams

In [ ]:
lr_bi = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_bi.fit(X_train_bi, y_train)
y_pred_bi = lr_bi.predict(X_val_bi)

# Evaluate and log metrics to outputs/results.csv
metrics_bi = evaluate_and_log(
    y_val, y_pred_bi,
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,2)",
    params="max_iter=1000, class_weight='balanced', random_state=42"
)

## 📊 5. Summary & Comparison

| Metric | TF-IDF (1,1) | TF-IDF (1,2) | Difference |
| :--- | :---: | :---: | :---: |
| **Vocabulary Size** | 12,575 | 58,182 | +45,607 (+362.7%) |
| **Accuracy** | 0.7685 | 0.7795 | +0.0110 |
| **Precision (Macro)** | 0.6914 | 0.7015 | +0.0102 |
| **Recall (Macro)** | 0.7015 | 0.7048 | +0.0033 |
| **F1-Score (Macro)** | 0.6957 | 0.7031 | +0.0074 |

### Core Observations:
1. **Vocabulary Expansion**: Adding bigrams exponentially increases the features. Bigrams capture multi-word sentiment indicators (e.g. "price cut", "earnings beat", "underperform needham").
2. **Baseline Boost**: Evaluating the F1-Score and general performance tells us how bigrams compare to unigram baselines under class balancing. Check `outputs/results.csv` for rolling leaderboard logs.